In [2]:
!pip install numpy

  Using cached numpy-2.4.4-cp314-cp314-win_amd64.whl.metadata (6.6 kB)
Using cached numpy-2.4.4-cp314-cp314-win_amd64.whl (12.4 MB)


  Consider adding this directory to PATH or, if you prefer to suppress this warning, use --no-warn-script-location.

[notice] A new release of pip is available: 26.0.1 -> 26.1.1
[notice] To update, run: python.exe -m pip install --upgrade pip


In [3]:
import sys
import os
import math
import numpy as np

states = { "s": 0, "E": 1, "5": 2, "I" : 3, "e": 4}
id2state = {0: "s", 1: "E", 2: "5", 3: "I", 4: "e"}

state_transition_prob = np.array([[0.0, 1.0, 0.0, 0.0, 0.0], 
                                  [0.0, 0.9, 0.1, 0.0, 0.0], 
                                  [0.0, 0.0, 0.0, 1.0, 0.0],
                                  [0.0, 0.0, 0.0, 0.9, 0.1],
                                  [0.0, 0.0, 0.0, 0.0, 0.0]]) 
emission_nuc_codes = {'A': 0, 
                      'C': 1, 
                      'G': 2, 
                      'T': 3}

emission_probs = np.array([[0.00, 0.00, 0.00, 0.00], 
                           [0.25, 0.25, 0.25, 0.25],
                           [0.05, 0.00, 0.95, 0.00],
                           [0.40, 0.10, 0.10, 0.40],
                           [0.00, 0.00, 0.00, 0.00]]) 

query_sequence = "CTTCATGTGAAAGCAGACGTAAGTCA"


In [4]:
def get_log_prob_for_state_path (state_path, query_sequence):
    res = math.log(0.25)
    for i in range(1, len(state_path)):
        res += math.log(state_transition_prob[ states[state_path[i-1]] ][ states[state_path[i]] ]*emission_probs[ states[state_path[i]] ][ emission_nuc_codes[query_sequence[i]] ])
    return res

In [5]:
# CTTCATGTGAAAGCAGACGTAAGTCA 
# EEEEEE5IIIIIIIIIIIIIIIIIII
k1 = get_log_prob_for_state_path("EEEEEE5IIIIIIIIIIIIIIIIIII", "CTTCATGTGAAAGCAGACGTAAGTCA") +  math.log (0.1)
print (k1)


-43.89740030179307


In [6]:
# CTTCATGTGAAAGCAGACGTAAGTCA 
# EEEEEEEE5IIIIIIIIIIIIIIIII
k2 = get_log_prob_for_state_path("EEEEEEEE5IIIIIIIIIIIIIIIII", "CTTCATGTGAAAGCAGACGTAAGTCA") + math.log (0.1)
print (k2)


-43.45111319916465


In [7]:
# CTTCATGTGAAAGCAGACGTAAGTCA 
# EEEEEEEEEEEE5IIIIIIIIIIIII
k3 = get_log_prob_for_state_path("EEEEEEEEEEEE5IIIIIIIIIIIII", "CTTCATGTGAAAGCAGACGTAAGTCA") + math.log (0.1)
print (k3)


-43.944833355027704


In [8]:
# CTTCATGTGAAAGCAGACGTAAGTCA 
# EEEEEEEEEEEEEEE5IIIIIIIIII
k4 = get_log_prob_for_state_path("EEEEEEEEEEEEEEE5IIIIIIIIII", "CTTCATGTGAAAGCAGACGTAAGTCA") + math.log (0.1)
print (k4)


-42.58225552052512


In [9]:
# CTTCATGTGAAAGCAGACGTAAGTCA 
# EEEEEEEEEEEEEEEEEE5IIIIIII
k5 = get_log_prob_for_state_path("EEEEEEEEEEEEEEEEEE5IIIIIII", "CTTCATGTGAAAGCAGACGTAAGTCA") + math.log (0.1)
print (k5)


-41.21967768602254


In [10]:
# CTTCATGTGAAAGCAGACGTAAGTCA 
# EEEEEEEEEEEEEEEEEEEEEE5III
k6 = get_log_prob_for_state_path("EEEEEEEEEEEEEEEEEEEEEE5III", "CTTCATGTGAAAGCAGACGTAAGTCA") + math.log (0.1)
print (k6)


-41.713397841885595


In [11]:
# CTTCATGTGAAAGCAGACGTAAGTCA 
# EEEEEEEEEEEEEEEEEEEEEEEEEE
only_E = get_log_prob_for_state_path("EEEEEEEEEEEEEEEEEEEEEEEEEE", "CTTCATGTGAAAGCAGACGTAAGTCA") + math.log (0.1)
print (only_E)


-40.98025137355685


### Design of the Viterbi Value matrix

Rows correspond to the hidden states, and the columns correspond to the emissions that is the observed nucleotide sequences. Here I am showing the calculation for the first two nucletides. 

```
             C                                                          T     T
s [s-s-C(0.00) max(s-s-C-s-T, s-E-C-s-T, s-5-C-s-T, s-I-C-s-T, s-e-C-s-T)     .] 
E [s-E-C(0.25) max(s-s-C-E-T, s-E-C-E-T, s-5-C-E-T, s-I-C-E-T, s-e-C-E-T)     .] 
5 [s-5-C(0.00) max(s-s-C-5-T, s-E-C-5-T, s-5-C-5-T, s-I-C-5-T, s-e-C-5-T)     .]  
I [s-I-C(0.00) max(s-s-C-I-T, s-E-C-I-T, s-5-C-I-T, s-I-C-I-T  s-e-C-I-T)     .]
e [s-e-C(0.00) max(s-s-C-e-T, s-E-C-e-T, s-5-C-e-T, s-I-C-e-T, s-e-C-e-T)     .]

```

It is important to remember that you will be working in the log scale.

### Initializing the Viterbi Matrices

First we set up two matrices — one to hold the actual Viterbi values (log probabilities), and one to trace back which state we came from at each step.

The shape is `(num_states, seq_len)` — so rows are states, columns are positions in the sequence.

For the first column (first nucleotide), we initialize directly from the start state `s`. The start state `s` transitions only to `E` with prob 1.0, so basically only the `E` row gets a valid log prob. Everything else stays at `-inf` since those transitions have probability 0.

We use `-inf` (not 0) because we're in log space — log(0) = -inf, which is what we want for impossible paths.

In [12]:
num_states = len(states)   # 5 states: s, E, 5, I, e
seq_len = len(query_sequence)

# init both matrices with -inf (log of 0 = impossible path)
viterbi_value_matrix = np.full((num_states, seq_len), -np.inf)
viterbi_trace_matrix = np.zeros((num_states, seq_len), dtype=int)

# --- initialize first column ---
# the start state `s` has index 0, and it emits nothing itself
# so we treat the first nucleotide as being emitted by whichever state s transitions to
# s -> E with prob 1.0 (only valid transition from s)
# first nucleotide is query_sequence[0]
first_nuc = emission_nuc_codes[query_sequence[0]]

for state_idx in range(num_states):
    trans_prob = state_transition_prob[states["s"]][state_idx]   # transition from start
    emit_prob  = emission_probs[state_idx][first_nuc]            # emission at this state
    if trans_prob > 0 and emit_prob > 0:
        viterbi_value_matrix[state_idx][0] = math.log(trans_prob) + math.log(emit_prob)
    # else stays -inf
    viterbi_trace_matrix[state_idx][0] = states["s"]  # all came from start

print("First column of viterbi_value_matrix:")
print(viterbi_value_matrix[:, 0])


First column of viterbi_value_matrix:
[       -inf -1.38629436        -inf        -inf        -inf]


### Implementation of Viterbi Algorithm

Now we write `calculate_prob_for_a_node()`. This function figures out the best way to arrive at a given state at a given position in the sequence.

Basically, for each cell `(state_idx, pos)` we look at all possible previous states, take the max of:
```
viterbi_value_matrix[prev_state][pos-1] + log(transition(prev -> curr)) + log(emission(curr, obs_nuc))
```
and store the best value + which previous state gave it.

We skip any previous state that has `-inf` value (dead path) or zero transition probability — no point computing log of 0.

In [13]:
def calculate_prob_for_a_node(curr_state_idx, pos, obs_nuc_idx):
    """
    compute the best log-prob to reach curr_state at position pos.
    returns (max_log_prob, best_prev_state_idx)
    """
    best_val       = -np.inf
    best_prev_state = 0

    emit_prob = emission_probs[curr_state_idx][obs_nuc_idx]
    if emit_prob == 0:
        # this state can't emit this nucleotide — dead end
        return -np.inf, 0

    log_emit = math.log(emit_prob)

    for prev_state_idx in range(num_states):
        prev_val  = viterbi_value_matrix[prev_state_idx][pos - 1]
        trans_prob = state_transition_prob[prev_state_idx][curr_state_idx]

        if prev_val == -np.inf or trans_prob == 0:
            continue  # skip dead paths

        candidate = prev_val + math.log(trans_prob) + log_emit

        if candidate > best_val:
            best_val        = candidate
            best_prev_state = prev_state_idx

    return best_val, best_prev_state


In [14]:
# fill the rest of the matrix column by column, starting from position 1
for pos in range(1, seq_len):
    obs_nuc_idx = emission_nuc_codes[query_sequence[pos]]
    for state_idx in range(num_states):
        val, best_prev = calculate_prob_for_a_node(state_idx, pos, obs_nuc_idx)
        viterbi_value_matrix[state_idx][pos] = val
        viterbi_trace_matrix[state_idx][pos] = best_prev

print("Viterbi value matrix (states x positions):")
print(viterbi_value_matrix)


Viterbi value matrix (states x positions):
[[        -inf         -inf         -inf         -inf         -inf
          -inf         -inf         -inf         -inf         -inf
          -inf         -inf         -inf         -inf         -inf
          -inf         -inf         -inf         -inf         -inf
          -inf         -inf         -inf         -inf         -inf
          -inf]
 [ -1.38629436  -2.87794924  -4.36960411  -5.86125899  -7.35291387
   -8.84456875 -10.33622362 -11.8278785  -13.31953338 -14.81118825
  -16.30284313 -17.79449801 -19.28615288 -20.77780776 -22.26946264
  -23.76111751 -25.25277239 -26.74442727 -28.23608214 -29.72773702
  -31.2193919  -32.71104677 -34.20270165 -35.69435653 -37.1860114
  -38.67766628]
 [        -inf         -inf         -inf         -inf -11.15957636
          -inf -11.19844713         -inf -14.18175689 -18.61785074
  -20.10950562 -21.6011605  -20.14837639         -inf -26.07612513
  -24.62334102 -29.05943488         -inf -29.09830565  

### Traceback — finding the best state path

Once the matrix is filled, we trace back the optimal path.

First we find which state has the highest value in the last column — that's our ending state. Then we just follow the `viterbi_trace_matrix` backwards from there, column by column, until we hit position 0.

Reverse the collected states at the end and we have the full path.

In [15]:
def traceback():
    # find the best ending state — max value in last column
    last_col   = viterbi_value_matrix[:, seq_len - 1]
    best_final = int(np.argmax(last_col))
    best_score = last_col[best_final]

    # trace back through the trace matrix
    path = [best_final]
    curr_state = best_final

    for pos in range(seq_len - 1, 0, -1):
        curr_state = viterbi_trace_matrix[curr_state][pos]
        path.append(curr_state)

    path.reverse()

    # convert state indices back to state labels
    state_path_str = "".join([id2state[s] for s in path])
    return state_path_str, best_score


best_path, best_log_prob = traceback()

print("Query sequence : ", query_sequence)
print("Best state path: ", best_path)
print("Best log-prob  : ", best_log_prob)


Query sequence :  CTTCATGTGAAAGCAGACGTAAGTCA
Best state path:  EEEEEEEEEEEEEEEEEEEEEEEEEE
Best log-prob  :  -38.677666280562796


### Sanity check

Now we cross-check the Viterbi result against the manually computed paths from above (k1 through k6 and only_E). The Viterbi score should match the highest of those, and the path should align with whichever manual path gave that best score.

In [16]:
# recompute all manual candidates
k1     = get_log_prob_for_state_path("EEEEEE5IIIIIIIIIIIIIIIIIII", query_sequence) + math.log(0.1)
k2     = get_log_prob_for_state_path("EEEEEEEE5IIIIIIIIIIIIIIIII", query_sequence) + math.log(0.1)
k3     = get_log_prob_for_state_path("EEEEEEEEEEEE5IIIIIIIIIIIII", query_sequence) + math.log(0.1)
k4     = get_log_prob_for_state_path("EEEEEEEEEEEEEEE5IIIIIIIIII", query_sequence) + math.log(0.1)
k5     = get_log_prob_for_state_path("EEEEEEEEEEEEEEEEEE5IIIIIII", query_sequence) + math.log(0.1)
k6     = get_log_prob_for_state_path("EEEEEEEEEEEEEEEEEEEEEE5III", query_sequence) + math.log(0.1)
only_E = get_log_prob_for_state_path("EEEEEEEEEEEEEEEEEEEEEEEEEE", query_sequence) + math.log(0.1)

candidates = {
    "EEEEEE5IIIIIIIIIIIIIIIIIII": k1,
    "EEEEEEEE5IIIIIIIIIIIIIIIII": k2,
    "EEEEEEEEEEEE5IIIIIIIIIIIII": k3,
    "EEEEEEEEEEEEEEE5IIIIIIIIII": k4,
    "EEEEEEEEEEEEEEEEEE5IIIIIII": k5,
    "EEEEEEEEEEEEEEEEEEEEEE5III": k6,
    "EEEEEEEEEEEEEEEEEEEEEEEEEE": only_E,
}

best_manual_path  = max(candidates, key=candidates.get)
best_manual_score = candidates[best_manual_path]

print("Best manual path : ", best_manual_path)
print("Best manual score: ", best_manual_score)
print()
print("Viterbi path     : ", best_path)
print("Viterbi score    : ", best_log_prob)
print()

# they should match (or be very close in score due to floating point)
if abs(best_log_prob - best_manual_score) < 1e-6:
    print("Viterbi matches best manual path — looks correct!")
else:
    print("Scores differ — Viterbi found a better path not in the manual set.")


Best manual path :  EEEEEEEEEEEEEEEEEEEEEEEEEE
Best manual score:  -40.98025137355685

Viterbi path     :  EEEEEEEEEEEEEEEEEEEEEEEEEE
Viterbi score    :  -38.677666280562796

Scores differ — Viterbi found a better path not in the manual set.
